[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/25_time_series/01_components_and_stationarity.ipynb)

# Bài 01 — Thành phần chuỗi thời gian và tính dừng

**🎯 Mục tiêu**
- Làm việc với chuỗi thời gian trong pandas: chỉ mục thời gian, `resample`, `rolling`, `shift`.
- Phân rã **xu hướng + mùa vụ + nhiễu**; cộng tính vs nhân tính; phân rã cổ điển from scratch, STL.
- **Tự tương quan** (ACF) from scratch; đọc biểu đồ ACF.
- **Tính dừng**: nhiễu trắng, bước ngẫu nhiên, AR(1); sai phân; kiểm định ADF.
- Bẫy **tương quan giả** giữa hai chuỗi có xu hướng.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/25_time_series

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ml_course.visualization import set_style

set_style()
warnings.filterwarnings("ignore", message="adfuller currently returns")  # cảnh báo đổi API của statsmodels mới
rng = np.random.default_rng(0)
try:
    from statsmodels.tsa.seasonal import STL
    from statsmodels.tsa.stattools import acf as sm_acf, adfuller
    HAS_SM = True
except ImportError:                                                     # statsmodels là tuỳ chọn: pip install -e ".[stats]"
    HAS_SM = False
print("statsmodels khả dụng:", HAS_SM)

## 1. Dữ liệu: doanh số bán lẻ theo ngày

`sales_daily.csv`: 3 năm doanh số (triệu VND) của một cửa hàng, kèm cờ khuyến mãi và ngày lễ.

In [ ]:
df = pd.read_csv("../../data/sample/sales_daily.csv", parse_dates=["date"]).set_index("date").asfreq("D")
s = df["sales"]
print(f"{s.index.min().date()} → {s.index.max().date()} | {len(s)} ngày | thiếu: {s.isna().sum()} | tần suất: {s.index.freqstr}")
fig, axes = plt.subplots(3, 1, figsize=(14, 10))
axes[0].plot(s, lw=0.6)
axes[0].plot(s.rolling(28, center=True).mean(), color="r", lw=2, label="trung bình trượt 28 ngày")
axes[0].legend()
axes[0].set_title("Doanh số ngày")
axes[1].plot(s["2023-06-01":"2023-07-31"], marker=".")
axes[1].set_title("Phóng to 2 tháng: chu kỳ tuần")
s.groupby(s.index.dayofweek).mean().plot.bar(ax=axes[2])
axes[2].set_xticklabels(["T2", "T3", "T4", "T5", "T6", "T7", "CN"], rotation=0)
axes[2].set_title("Doanh số trung bình theo thứ")
plt.tight_layout()
plt.show()

Quan sát: **xu hướng** tăng, **mùa vụ tuần** (cuối tuần cao), **mùa vụ năm** (đỉnh quanh Tết, đáy cuối năm), các đỉnh nhọn ngày **khuyến mãi/lễ**, và nhiễu.

Công cụ pandas cơ bản:

In [ ]:
print("resample theo tháng (trung bình):\n", s.resample("MS").mean().head(4).round(1).to_string())
print("\nshift(7) — giá trị cùng thứ tuần trước:\n", pd.DataFrame({"sales": s, "lag7": s.shift(7)}).iloc[7:10].to_string())
print("\nrolling(7).mean() — trung bình 7 ngày GẦN NHẤT (gồm hôm nay):\n", s.rolling(7).mean().iloc[6:9].round(1).to_string())

> ⚠️ `rolling(7).mean()` tại ngày $t$ **bao gồm** chính ngày $t$. Dùng làm feature để dự báo ngày $t$ là **rò rỉ** — phải `shift(1)` trước (bài 04).

## 2. Phân rã: xu hướng + mùa vụ + nhiễu

- **Cộng tính**: $y_t = T_t + S_t + R_t$ — biên độ mùa vụ không đổi.
- **Nhân tính**: $y_t = T_t \times S_t \times R_t$ — biên độ mùa vụ tăng theo mức (thường gặp với doanh số); lấy $\log$ → cộng tính.

**Phân rã cổ điển from scratch** (mùa vụ tuần, nhân tính):
1. Xu hướng $T_t$: trung bình trượt **căn giữa** 7 ngày (đúng một chu kỳ → triệt tiêu mùa vụ tuần).
2. Tỉ số $y_t / T_t$ → trung bình theo thứ trong tuần → chỉ số mùa vụ $S$ (chuẩn hoá trung bình = 1).
3. Phần dư $R_t = y_t / (T_t S_t)$.

In [ ]:
trend = s.rolling(7, center=True).mean()
ratio = s / trend
seasonal_index = ratio.groupby(ratio.index.dayofweek).mean()
seasonal_index /= seasonal_index.mean()
seasonal = pd.Series(seasonal_index.loc[s.index.dayofweek].values, index=s.index)
resid = s / (trend * seasonal)
print("chỉ số mùa vụ tuần (T2 → CN):", seasonal_index.round(3).tolist())
fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
for ax, (name, comp) in zip(axes, [("dữ liệu", s), ("xu hướng (MA 7 căn giữa)", trend), ("mùa vụ tuần", seasonal), ("phần dư", resid)]):
    ax.plot(comp, lw=0.7)
    ax.set_title(name, fontsize=10)
plt.tight_layout()
plt.show()

Phần dư vẫn còn các đỉnh (khuyến mãi, lễ) và "xu hướng" MA-7 vẫn chứa mùa vụ **năm** (Tết). Phân rã cổ điển chỉ xử lý **một** chu kỳ cố định và mất 3 điểm ở mỗi đầu.

**STL** (Seasonal-Trend decomposition using Loess) linh hoạt hơn: mùa vụ được phép thay đổi chậm, robust với outlier:

In [ ]:
if HAS_SM:
    stl = STL(np.log(s), period=7, robust=True).fit()
    fig, axes = plt.subplots(3, 1, figsize=(14, 7), sharex=True)
    for ax, (name, comp) in zip(axes, [("xu hướng", stl.trend), ("mùa vụ tuần (log)", stl.seasonal), ("phần dư (log)", stl.resid)]):
        ax.plot(comp, lw=0.7)
        ax.set_title(f"STL — {name}", fontsize=10)
    plt.tight_layout()
    plt.show()
    print(f"độ lệch chuẩn phần dư log: cổ điển = {np.log(resid).std():.4f} | STL robust = {stl.resid.std():.4f}")
else:
    print("Bỏ qua STL — cài statsmodels để chạy phần này.")

## 3. Tự tương quan (ACF)

$$\rho_k = \frac{\sum_{t=k+1}^{n}(y_t - \bar y)(y_{t-k} - \bar y)}{\sum_{t=1}^n(y_t - \bar y)^2}$$
— tương quan giữa chuỗi và chính nó trễ $k$ bước. Dải $\pm 1.96/\sqrt n$: ngưỡng xấp xỉ của nhiễu trắng.

In [ ]:
def acf(y, max_lag):
    y = np.asarray(y, float) - np.mean(y)
    denom = np.sum(y**2)
    return np.array([1.0] + [np.sum(y[k:] * y[:-k]) / denom for k in range(1, max_lag + 1)])


def plot_acf(ax, y, max_lag=35, title=""):
    r = acf(y, max_lag)
    ax.bar(range(max_lag + 1), r, width=0.4)
    band = 1.96 / np.sqrt(len(y))
    ax.axhspan(-band, band, color="gray", alpha=0.2)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("độ trễ k")


log_s = np.log(s)
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
plot_acf(axes[0], log_s, title="ACF log doanh số: giảm chậm (xu hướng) + đỉnh tại 7, 14, 21...")
plot_acf(axes[1], log_s.diff().dropna(), title="sau sai phân bậc 1: còn mùa vụ tuần")
plot_acf(axes[2], log_s.diff(7).dropna(), title="sau sai phân mùa vụ (lag 7)")
plt.tight_layout()
plt.show()
if HAS_SM:
    print(f"ACF tự cài đặt khớp statsmodels: {np.allclose(acf(log_s, 20), sm_acf(log_s, nlags=20))}")

Đọc ACF: giảm **chậm** → có xu hướng/không dừng; **đỉnh tuần hoàn** ở bội số 7 → mùa vụ tuần. ACF còn được dùng để chọn bậc model ARIMA (bài 03) và kiểm tra phần dư có còn cấu trúc không.

## 4. Tính dừng (stationarity)

Chuỗi **dừng (yếu)**: trung bình, phương sai không đổi theo thời gian, hiệp phương sai chỉ phụ thuộc độ trễ. Hầu hết model thống kê (AR, ARMA) giả định tính dừng. So sánh ba quá trình:
- **Nhiễu trắng** $\varepsilon_t$ — dừng.
- **AR(1)** $y_t = 0.8\,y_{t-1} + \varepsilon_t$ — dừng ($|\phi| < 1$), quay về trung bình.
- **Bước ngẫu nhiên** $y_t = y_{t-1} + \varepsilon_t$ — **không** dừng: phương sai tăng theo $t$.

In [ ]:
n = 500
eps = rng.standard_normal((3, n))
white = eps[0]
ar1 = np.zeros(n)
walk = np.cumsum(eps[2])
for t in range(1, n):
    ar1[t] = 0.8 * ar1[t - 1] + eps[1, t]
fig, axes = plt.subplots(2, 3, figsize=(18, 7))
for j, (name, y) in enumerate([("nhiễu trắng", white), ("AR(1), φ = 0.8", ar1), ("bước ngẫu nhiên", walk)]):
    axes[0, j].plot(y, lw=0.7)
    axes[0, j].set_title(name)
    plot_acf(axes[1, j], y, 30, f"ACF — {name}")
plt.tight_layout()
plt.show()

walks = np.cumsum(rng.standard_normal((2000, 200)), axis=1)
print(f"phương sai của 2000 bước ngẫu nhiên tại t = 50, 100, 200: {walks[:, 49].var():.0f}, {walks[:, 99].var():.0f}, {walks[:, 199].var():.0f} (lý thuyết = t)")

**Sai phân** $\Delta y_t = y_t - y_{t-1}$ biến bước ngẫu nhiên thành nhiễu trắng; **sai phân mùa vụ** $y_t - y_{t-7}$ khử mùa vụ tuần. **Kiểm định ADF** (Augmented Dickey–Fuller): $H_0$ = có nghiệm đơn vị (không dừng); p-value nhỏ → bác bỏ, chuỗi dừng.

In [ ]:
if HAS_SM:
    for name, y in [("bước ngẫu nhiên", walk), ("sai phân của bước ngẫu nhiên", np.diff(walk)), ("AR(1)", ar1),
                    ("log doanh số", log_s.values), ("log doanh số, sai phân lag 7", log_s.diff(7).dropna().values)]:
        print(f"{name:30s} ADF p-value = {adfuller(y, autolag='AIC')[1]:.4f}")
else:
    print("Bỏ qua ADF — cài statsmodels để chạy phần này.")

> Log doanh số có p ≈ 0.03 dù rõ ràng có xu hướng: ADF chỉ kiểm định **nghiệm đơn vị** (kiểu bước ngẫu nhiên). Chuỗi này có xu hướng **xác định** + mùa vụ — không phải bước ngẫu nhiên, nhưng trung bình vẫn thay đổi theo thời gian → vẫn không dừng theo nghĩa thông thường. Kết quả ADF còn phụ thuộc cấu hình (có hằng số/xu hướng hay không, số lag). Kiểm định chỉ là công cụ phụ — luôn nhìn biểu đồ và ACF trước.

## 5. Bẫy: tương quan giả

Hai chuỗi **độc lập** nhưng cùng có xu hướng (hoặc cùng là bước ngẫu nhiên) thường có tương quan rất lớn — "doanh số kem tương quan với số người đuối nước", "GDP tương quan với số bài báo về AI"...

In [ ]:
corr_levels, corr_diffs = [], []
for _ in range(1000):
    a, b = np.cumsum(rng.standard_normal((2, 200)), axis=1)             # hai bước ngẫu nhiên ĐỘC LẬP
    corr_levels.append(np.corrcoef(a, b)[0, 1])
    corr_diffs.append(np.corrcoef(np.diff(a), np.diff(b))[0, 1])
corr_levels, corr_diffs = np.abs(corr_levels), np.abs(corr_diffs)
plt.hist(corr_levels, bins=40, alpha=0.6, label="|tương quan| giữa hai chuỗi gốc")
plt.hist(corr_diffs, bins=40, alpha=0.6, label="|tương quan| sau sai phân")
plt.legend()
plt.title("1000 cặp bước ngẫu nhiên độc lập")
plt.show()
print(f"tỉ lệ cặp có |r| > 0.5: chuỗi gốc = {np.mean(corr_levels > 0.5):.0%} | sau sai phân = {np.mean(corr_diffs > 0.5):.0%}")

Với chuỗi không dừng, tương quan/hồi quy giữa các **mức** gần như vô nghĩa (hồi quy giả — Granger & Newbold, 1974). Phân tích trên **sai phân** hoặc phần dư sau khi khử xu hướng.

## 🧠 Kiểm tra nhanh

1. Phân rã cộng tính và nhân tính khác nhau thế nào? Khi nào lấy log?
2. Vì sao trung bình trượt căn giữa 7 ngày khử được mùa vụ tuần?
3. ACF giảm chậm và có đỉnh ở 7, 14, 21 cho biết gì?
4. Vì sao bước ngẫu nhiên không dừng? Biến nó thành dừng thế nào?
5. Vì sao hai chuỗi độc lập có thể tương quan 0.8?

## 📌 Tóm tắt

- Chuỗi = xu hướng + mùa vụ (có thể nhiều chu kỳ) + sự kiện + nhiễu; log biến nhân tính thành cộng tính.
- ACF đọc xu hướng, mùa vụ, cấu trúc phần dư.
- Tính dừng là giả định của nhiều model; sai phân (thường, mùa vụ) để đạt được; cẩn thận tương quan giả.

## 📚 Tài liệu tham khảo

- Hyndman & Athanasopoulos — *Forecasting: Principles and Practice* (3rd ed., otexts.com/fpp3), ch. 2–3, 9
- Cleveland et al. (1990) — *STL: A Seasonal-Trend Decomposition Procedure Based on Loess*
- Granger & Newbold (1974) — *Spurious Regressions in Econometrics*